# YOLO26 Garbage Classification: Training + PTQ

This notebook uses the local image-folder dataset at `~/Downloads/Garbage Dataset - Post-Training Quantization (PTQ)/standardized_256`. It creates a reproducible 70/15/15 train/validation/test split beside the downloaded dataset, trains `yolo26n-cls`, evaluates on the held-out test set, and exports an INT8 ONNX model calibrated with validation images.

Run cells from top to bottom. The source dataset is left unchanged. The split uses hard links where Windows permits and falls back to copying. Training downloads the pretrained checkpoint on first use; for a fully offline run, set `MODEL_NAME` to a compatible local YOLO26 classification `.pt` checkpoint. The current kernel has Ultralytics installed; for a fresh kernel, run `%pip install ultralytics` and restart it.

## Enable the RTX 4050

Windows detects the NVIDIA GeForce RTX 4050 Laptop GPU. The notebook uses CUDA-enabled PyTorch, and the verification cell should report the GPU. If setting up a fresh environment, run the next cell once, then use **Restart Kernel** in VS Code before running the verification cell.

In [7]:
%pip install --upgrade torch==2.14.0+cu130 torchvision==0.29.0+cu130 --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path

import torch
import ultralytics

WORKSPACE = Path.cwd()
DATASET_ROOT = Path.home() / "Downloads" / "Garbage Dataset - Post-Training Quantization (PTQ)"
SOURCE_DATA = DATASET_ROOT / "standardized_256"
SPLIT_DATA = DATASET_ROOT / "yolo_cls_256_split"
MODEL_NAME = "yolo26n-cls.pt"
IMAGE_SIZE = 224
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
RANDOM_SEED = 42
EPOCHS = 30
BATCH_SIZE = 16
DEVICE = 0 if torch.cuda.is_available() else "cpu"

if not SOURCE_DATA.is_dir():
    raise FileNotFoundError(f"Dataset folder not found: {SOURCE_DATA}")

CLASS_NAMES = sorted(path.name for path in SOURCE_DATA.iterdir() if path.is_dir())
if len(CLASS_NAMES) != 10:
    raise ValueError(f"Expected 10 class folders, found {len(CLASS_NAMES)}: {CLASS_NAMES}")

image_counts = {
    class_name: sum(1 for path in (SOURCE_DATA / class_name).iterdir() if path.is_file())
    for class_name in CLASS_NAMES
}
print(f"Ultralytics: {ultralytics.__version__}")
print(f"PyTorch: {torch.__version__}; CUDA build: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Device: {'CUDA GPU' if DEVICE == 0 else 'CPU'}")
print(f"Dataset: {SOURCE_DATA}")
print(f"Classes: {CLASS_NAMES}")
print(f"Total images: {sum(image_counts.values()):,}")
print(f"Images per class: {image_counts}")
print(f"Split: {1 - VALIDATION_FRACTION - TEST_FRACTION:.0%} train / {VALIDATION_FRACTION:.0%} val / {TEST_FRACTION:.0%} test")
print(f"Split output: {SPLIT_DATA}")

Ultralytics: 8.4.163
PyTorch: 2.14.0+cu130; CUDA build: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Device: CUDA GPU
Dataset: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\standardized_256
Classes: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']
Total images: 12,259
Images per class: {'battery': 756, 'biological': 699, 'cardboard': 1411, 'clothes': 1892, 'glass': 1736, 'metal': 930, 'paper': 1336, 'plastic': 1597, 'shoes': 1449, 'trash': 453}
Split: 70% train / 15% val / 15% test
Split output: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split


In [16]:
import json
import os
import random
import shutil
import tempfile

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_MANIFEST = SPLIT_DATA / "split_manifest.json"

if SPLIT_DATA.exists():
    if not SPLIT_MANIFEST.is_file():
        raise FileExistsError(f"Refusing to overwrite existing unrecognized folder: {SPLIT_DATA}")
    manifest = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
    if manifest.get("source") != str(SOURCE_DATA) or manifest.get("classes") != CLASS_NAMES:
        raise ValueError(f"Existing split does not match this source dataset: {SPLIT_DATA}")
    print(f"Using existing split at {SPLIT_DATA}")
else:
    SPLIT_DATA.parent.mkdir(parents=True, exist_ok=True)
    build_root = Path(tempfile.mkdtemp(prefix="garbage_cls_split_", dir=SPLIT_DATA.parent))
    rng = random.Random(RANDOM_SEED)
    split_counts = {split: {} for split in ("train", "val", "test")}
    linked_files = 0
    copied_files = 0

    try:
        for class_name in CLASS_NAMES:
            class_files = sorted(
                path for path in (SOURCE_DATA / class_name).iterdir()
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            )
            if len(class_files) < 3:
                raise ValueError(f"Class {class_name!r} needs at least 3 images")

            rng.shuffle(class_files)
            test_count = max(1, round(len(class_files) * TEST_FRACTION))
            val_count = max(1, round(len(class_files) * VALIDATION_FRACTION))
            train_end = len(class_files) - test_count - val_count
            partitions = {
                "train": class_files[:train_end],
                "val": class_files[train_end:len(class_files) - test_count],
                "test": class_files[len(class_files) - test_count:],
            }

            for split, files in partitions.items():
                destination = build_root / split / class_name
                destination.mkdir(parents=True, exist_ok=True)
                split_counts[split][class_name] = len(files)
                for source_file in files:
                    destination_file = destination / source_file.name
                    try:
                        os.link(source_file, destination_file)
                        linked_files += 1
                    except OSError:
                        shutil.copy2(source_file, destination_file)
                        copied_files += 1

        manifest = {
            "source": str(SOURCE_DATA),
            "seed": RANDOM_SEED,
            "validation_fraction": VALIDATION_FRACTION,
            "test_fraction": TEST_FRACTION,
            "classes": CLASS_NAMES,
            "counts": split_counts,
        }
        (build_root / "split_manifest.json").write_text(
            json.dumps(manifest, indent=2), encoding="utf-8"
        )
        build_root.rename(SPLIT_DATA)
    except Exception:
        shutil.rmtree(build_root, ignore_errors=True)
        raise

    print(f"Created split at {SPLIT_DATA}")
    print(f"Hard-linked {linked_files:,} images; copied {copied_files:,} images")
    print(f"Per-split totals: { {split: sum(counts.values()) for split, counts in split_counts.items()} }")

Using existing split at C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split


## Train

The first training run needs the pretrained checkpoint to be available locally (Ultralytics downloads `yolo26n-cls.pt` if it is not cached). This cell defines the run; the following cell starts training. On CPU, 30 epochs can take a while. Change `EPOCHS`, `BATCH_SIZE`, or `MODEL_NAME` in the configuration cell before starting.

In [17]:
from ultralytics import YOLO

TRAIN_ARGS = {
    "data": str(SPLIT_DATA),
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "device": DEVICE,
    "workers": 0,
    "patience": 8,
    "project": str(WORKSPACE / "runs" / "classify"),
    "name": "garbage_yolo26n_cls",
}


def train_classifier():
    classifier = YOLO(MODEL_NAME)
    classifier.train(**TRAIN_ARGS)
    trainer = classifier.trainer
    if trainer is None or trainer.save_dir is None:
        raise RuntimeError("Ultralytics finished without a training output directory")
    best_model_path = Path(trainer.save_dir) / "weights" / "best.pt"
    if not best_model_path.is_file():
        raise FileNotFoundError(f"Training did not produce best.pt at {best_model_path}")
    return best_model_path

print(f"Training configuration: {TRAIN_ARGS}")

Training configuration: {'data': 'C:\\Users\\nitis\\Downloads\\Garbage Dataset - Post-Training Quantization (PTQ)\\yolo_cls_256_split', 'epochs': 30, 'imgsz': 224, 'batch': 32, 'device': 'cpu', 'workers': 0, 'patience': 8, 'project': 'c:\\Users\\nitis\\YOLO26\\runs\\classify', 'name': 'garbage_yolo26n_cls'}


In [18]:
BEST_MODEL_PATH = train_classifier()
print(f"Best checkpoint: {BEST_MODEL_PATH}")

Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cpu CPU (AMD Ryzen 7 7445HS w/ Radeon 740M Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentu

Matplotlib is building the font cache; this may take a moment.


       1/30         0G      1.224          5        224: 100% ━━━━━━━━━━━━ 269/269 1.2it/s 3:490.7ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 29/29 1.9it/s 15.0s0.5s
                   all      0.836      0.991

      Epoch    GPU_mem       loss  Instances       Size
       2/30         0G      0.519          5        224: 100% ━━━━━━━━━━━━ 269/269 1.5it/s 3:040.3ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 29/29 2.2it/s 13.0s0.4s
                   all      0.884      0.995

      Epoch    GPU_mem       loss  Instances       Size
       3/30         0G     0.4744          5        224: 100% ━━━━━━━━━━━━ 269/269 1.4it/s 3:160.3ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 29/29 2.1it/s 13.5s0.5s
                   all      0.867      0.992

      Epoch    GPU_mem       loss  Instances       Size
       4/30         0G     0.4439          5        224: 100% ━━━━━━━━━━━━ 269/269 1.5it/s 2:570.4ss
               classes   t

## Evaluate the FP32 model

After training finishes, run this section to measure top-1 and top-5 accuracy on the untouched test split.

In [11]:
def evaluate_classifier(model_path, split):
    classifier = YOLO(str(model_path))
    metrics = classifier.val(
        data=str(SPLIT_DATA),
        split=split,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=0,
    )
    return float(metrics.top1), float(metrics.top5)

print("Test evaluation helper is ready.")

Test evaluation helper is ready.


In [19]:
FP32_TOP1, FP32_TOP5 = evaluate_classifier(BEST_MODEL_PATH, "test")
print(f"FP32 test accuracy: top-1={FP32_TOP1:.4f}, top-5={FP32_TOP5:.4f}")

Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cpu CPU (AMD Ryzen 7 7445HS w/ Radeon 740M Graphics)
YOLO26n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 3.2 GFLOPs
train: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\train... found 8581 images in 10 classes  
val: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\val... found 1839 images in 10 classes  
test: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\test... found 1839 images in 10 classes  
WARNING test: Slow image access detected (ping: 0.10.0 ms, read: 1.70.6 MB/s, size: 14.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
test: Scanning C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\test... 1839 images, 0 corrupt: 100% ━━━━━━

## Export INT8 ONNX with PTQ

This uses 25% of the validation split as representative calibration data. The test split is not used for calibration, so it remains a fair final comparison.

In [12]:
def export_int8_onnx(model_path):
    classifier = YOLO(str(model_path))
    exported_path = classifier.export(
        format="onnx",
        quantize=8,
        data=str(SPLIT_DATA),
        split="val",
        fraction=0.25,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
    )
    if not exported_path:
        raise RuntimeError("Ultralytics did not return an exported ONNX path")
    onnx_path = Path(str(exported_path))
    if not onnx_path.is_file():
        raise FileNotFoundError(f"INT8 ONNX export not found: {onnx_path}")
    return onnx_path

print("INT8 ONNX PTQ export helper is ready.")

INT8 ONNX PTQ export helper is ready.


In [20]:
PTQ_MODEL_PATH = export_int8_onnx(BEST_MODEL_PATH)
print(f"INT8 ONNX model: {PTQ_MODEL_PATH}")

Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cpu CPU (AMD Ryzen 7 7445HS w/ Radeon 740M Graphics)
YOLO26n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 0.4 GFLOPs

PyTorch: starting from 'C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 10) (3.1 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.10 environment at: c:\Users\nitis\AppData\Local\Programs\Python\Python312
Resolved 12 packages in 434ms
 Downloaded onnx
 Downloaded onnxruntime
Prepared 7 packages in 4.24s
Installed 7 packages in 774ms
 + colorama==0.4.6
 + flatbuffers==25.12.19
 + ml-dtypes==0.6.0
 + onnx==1.23.0
 + onnxruntime==1.30.0
 + onnxslim==0.1.96
 + protobuf==7.36.2

requirements: AutoUpdate success  6.2s
WARNING requirements: Restart runtime or rerun command for updates to take effect


ONNX: startin

ONNX: export success  16.1s, saved as 'C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best_int8.onnx' (1.6 MB)

Export complete (16.2s)
Results saved to C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best_int8.onnx
Predict:         yolo predict task=classify model=C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best_int8.onnx imgsz=224 
Validate:        yolo val task=classify model=C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best_int8.onnx imgsz=224 data=C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split  
Visualize:       https://netron.app
INT8 ONNX model: C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best_int8.onnx


## Compare INT8 against FP32

Run after the export. Both accuracies below use the same held-out test split; calibration used only validation images.

In [21]:
INT8_TOP1, INT8_TOP5 = evaluate_classifier(PTQ_MODEL_PATH, "test")
FP32_SIZE_MB = BEST_MODEL_PATH.stat().st_size / (1024 ** 2)
INT8_SIZE_MB = PTQ_MODEL_PATH.stat().st_size / (1024 ** 2)

print(f"FP32 test accuracy: top-1={FP32_TOP1:.4f}, top-5={FP32_TOP5:.4f}")
print(f"INT8 test accuracy: top-1={INT8_TOP1:.4f}, top-5={INT8_TOP5:.4f}")
print(f"Top-1 accuracy change: {INT8_TOP1 - FP32_TOP1:+.4f}")
print(f"File sizes: FP32={FP32_SIZE_MB:.2f} MiB, INT8 ONNX={INT8_SIZE_MB:.2f} MiB")

Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cpu CPU (AMD Ryzen 7 7445HS w/ Radeon 740M Graphics)
Loading C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best_int8.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.30.0 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 224, 224)
train: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\train... found 8581 images in 10 classes  
val: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\val... found 1839 images in 10 classes  
test: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\test... found 1839 images in 10 classes  
test: Fast image access  (ping: 0.10.0 ms, read: 64.829.5 MB/s, size: 14.3 KB)
test: Scanning C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\test... 1839 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1839/1839  

## QAT Fine-Tune and Compare

This fine-tunes the trained FP32 checkpoint with fake INT8 quantization, then exports and tests the QAT checkpoint. The active kernel now detects the RTX 4050 using CUDA PyTorch. NVIDIA ModelOpt is required; run the QAT setup cell before training if it is not installed.

### Install NVIDIA ModelOpt (one time)

Run this cell once in a fresh environment, then use **Restart Kernel** in VS Code. The current environment already has ModelOpt installed.

In [8]:
%pip install --upgrade nvidia-modelopt huggingface-hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import torch
from ultralytics import YOLO

QAT_RUNS_DIR = Path.cwd() / "runs" / "classify"
QAT_SPLIT_DATA = Path.home() / "Downloads" / "Garbage Dataset - Post-Training Quantization (PTQ)" / "yolo_cls_256_split"
FP32_CANDIDATES = [
    path
    for path in QAT_RUNS_DIR.glob("garbage_yolo26n_cls*/weights/best.pt")
    if "qat" not in path.parents[1].name.lower()
]
if not FP32_CANDIDATES:
    raise FileNotFoundError(f"No trained FP32 checkpoint found under {QAT_RUNS_DIR}")
if not QAT_SPLIT_DATA.is_dir():
    raise FileNotFoundError(f"Prepared classification split not found: {QAT_SPLIT_DATA}")
QAT_SOURCE_MODEL = max(FP32_CANDIDATES, key=lambda path: path.stat().st_mtime)
QAT_IMAGE_SIZE = 224
QAT_BATCH_SIZE = 8
QAT_EPOCHS = 5
QAT_CUDA_AVAILABLE = torch.cuda.is_available()

QAT_ARGS = {
    "data": str(QAT_SPLIT_DATA),
    "quantize": 8,
    "epochs": QAT_EPOCHS,
    "imgsz": QAT_IMAGE_SIZE,
    "batch": QAT_BATCH_SIZE,
    "device": 0 if QAT_CUDA_AVAILABLE else "cpu",
    "optimizer": "AdamW",
    "lr0": 1e-5,
    "lrf": 0.1,
    "warmup_epochs": 0.5,
    "cos_lr": True,
    "mosaic": 0.0,
    "compile": False,
    "workers": 0,
    "project": str(QAT_RUNS_DIR),
    "name": "garbage_yolo26n_cls_qat",
}


def train_qat():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "QAT requires a CUDA-enabled NVIDIA GPU with Ultralytics ModelOpt support. "
            "This kernel has CPU-only PyTorch; run this cell in a CUDA environment."
        )

    qat_model = YOLO(str(QAT_SOURCE_MODEL))
    qat_model.train(**QAT_ARGS)
    trainer = qat_model.trainer
    if trainer is None or trainer.save_dir is None:
        raise RuntimeError("QAT finished without a training output directory")
    best_path = Path(trainer.save_dir) / "weights" / "best.pt"
    if not best_path.is_file():
        raise FileNotFoundError(f"QAT did not produce best.pt at {best_path}")
    return best_path

print(f"QAT source checkpoint: {QAT_SOURCE_MODEL}")
print(f"QAT dataset split: {QAT_SPLIT_DATA}")
print(f"CUDA available: {QAT_CUDA_AVAILABLE}")
print(f"QAT settings: {QAT_ARGS}")

QAT source checkpoint: c:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best.pt
QAT dataset split: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split
CUDA available: True
QAT settings: {'data': 'C:\\Users\\nitis\\Downloads\\Garbage Dataset - Post-Training Quantization (PTQ)\\yolo_cls_256_split', 'quantize': 8, 'epochs': 5, 'imgsz': 224, 'batch': 8, 'device': 0, 'optimizer': 'AdamW', 'lr0': 1e-05, 'lrf': 0.1, 'warmup_epochs': 0.5, 'cos_lr': True, 'mosaic': 0.0, 'compile': False, 'workers': 0, 'project': 'c:\\Users\\nitis\\YOLO26\\runs\\classify', 'name': 'garbage_yolo26n_cls_qat'}


### Run QAT Fine-Tuning

Run this on a CUDA-enabled NVIDIA GPU. It starts from the trained FP32 `best.pt`, fine-tunes for five epochs with INT8 fake quantization, and saves a QAT checkpoint in the classify runs folder. The low learning rate is intentional for fine-tuning.

In [3]:
QAT_BEST_MODEL_PATH = train_qat()
print(f"Best QAT checkpoint: {QAT_BEST_MODEL_PATH}")

Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=c:\Users\nitis\YOLO26\

c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\modelopt\torch\utils\import_utils.py:33: UserWarning: Your device is potentially capable to use triton kernel to speed up quantization simulations. Try to install triton with `pip install triton`.
  warn_rank_0(msg_if_missing)
c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
W0927 18:11:16.429000 33416 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Preparing INT8 quantization-aware training from 8 calibration batches...
Inserted 200 quantizers


c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\modelopt\torch\export\layer_utils.py:29: UserWarning: Cannot find transformers package. Hugginface modules cannot be exported.
  warn("Cannot find transformers package. Hugginface modules cannot be exported.")


AMP: running Automatic Mixed Precision (AMP) checks...
AMP: downloading yolo26n.pt for AMP checks (one-time, not used for training)...
AMP: checks passed 
train: Fast image access  (ping: 0.10.0 ms, read: 129.867.5 MB/s, size: 17.6 KB)
train: Scanning C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\train... 8581 images, 0 corrupt: 100% ━━━━━━━━━━━━ 8581/8581  0.0s
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 30.216.2 MB/s, size: 15.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\val... 1839 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1839/1839 3.6Kit/s 0.5s0.0s
val: New cache created: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\val.cache
optimizer: AdamW(lr=1e-05, momentum=0.93

W0927 18:11:37.058000 33416 site-packages\torch\utils\cpp_extension.py:611] Error checking compiler version for cl
W0927 18:11:37.058000 33416 site-packages\torch\utils\cpp_extension.py:611] Traceback (most recent call last):
W0927 18:11:37.058000 33416 site-packages\torch\utils\cpp_extension.py:611]   File "c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\cpp_extension.py", line 607, in get_compiler_abi_compatibility_and_version
W0927 18:11:37.058000 33416 site-packages\torch\utils\cpp_extension.py:611]     compiler_info = subprocess.check_output(compiler, stderr=subprocess.STDOUT)
W0927 18:11:37.058000 33416 site-packages\torch\utils\cpp_extension.py:611]                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
W0927 18:11:37.058000 33416 site-packages\torch\utils\cpp_extension.py:611]   File "c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 466, in check_output
W0927 18:11:37.058000 33416 site-


      Epoch    GPU_mem       loss  Instances       Size
        1/5     0.215G     0.3981          5        224: 100% ━━━━━━━━━━━━ 1073/1073 5.8it/s 3:060.2ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 115/115 6.9it/s 16.7s0.2s
                   all      0.928      0.997

      Epoch    GPU_mem       loss  Instances       Size
        2/5     0.215G     0.3062          5        224: 100% ━━━━━━━━━━━━ 1073/1073 6.3it/s 2:500.2ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 115/115 6.9it/s 16.6s0.1s
                   all      0.927      0.997

      Epoch    GPU_mem       loss  Instances       Size
        3/5     0.215G     0.2958          5        224: 100% ━━━━━━━━━━━━ 1073/1073 5.9it/s 3:020.1ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 115/115 7.8it/s 14.7s0.1s
                   all      0.925      0.997

      Epoch    GPU_mem       loss  Instances       Size
        4/5     0.215G     0.2836          5        224: 1

c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\modelopt\torch\quantization\nn\modules\quant_module.py:111: UserWarning: Could not identify the device for TensorQuantizer states of model.10.pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


YOLO26n-cls summary (fused): 128 layers, 1,538,834 parameters, 0 gradients, 3.2 GFLOPs
train: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\train... found 8581 images in 10 classes  
val: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\val... found 1839 images in 10 classes  
test: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\test... found 1839 images in 10 classes  
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 115/115 8.3it/s 13.8s0.1s
                   all      0.932      0.997
Speed: 0.5ms preprocess, 4.7ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls_qat-2
Inserted 200 quantizers
Best QAT checkpoint: C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls_qat-2\weights\best.pt


c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\modelopt\torch\quantization\nn\modules\quant_module.py:111: UserWarning: Could not identify the device for TensorQuantizer states of model.10.pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


### Export the QAT Model

The QAT checkpoint carries quantization ranges learned during fine-tuning, so this ONNX export uses those ranges rather than PTQ calibration data. Keep `quantize=8` for the INT8 ONNX artifact.

In [4]:
def export_qat_onnx(checkpoint_path):
    qat_model = YOLO(str(checkpoint_path))
    exported_path = qat_model.export(
        format="onnx",
        quantize=8,
        imgsz=QAT_IMAGE_SIZE,
        device=0,
    )
    if not exported_path:
        raise RuntimeError("Ultralytics did not return a QAT ONNX path")
    onnx_path = Path(str(exported_path))
    if not onnx_path.is_file():
        raise FileNotFoundError(f"QAT ONNX export not found: {onnx_path}")
    return onnx_path

print("QAT ONNX exporter is ready.")

QAT ONNX exporter is ready.


In [5]:
QAT_ONNX_PATH = export_qat_onnx(QAT_BEST_MODEL_PATH)
print(f"QAT INT8 ONNX model: {QAT_ONNX_PATH}")

Inserted 200 quantizers
Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO26n-cls summary (fused): 128 layers, 1,538,834 parameters, 0 gradients, 0.4 GFLOPs

PyTorch: starting from 'C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls_qat-2\weights\best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 10) (3.2 MB)

ONNX: starting export with onnx 1.23.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success  2.7s, saved as 'C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls_qat-2\weights\best.onnx' (6.0 MB)

Export complete (3.4s)
Results saved to C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls_qat-2\weights\best.onnx
Predict:         yolo predict task=classify model=C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls_qat-2\weights\best.onnx imgsz=224 
Validate:        yolo val task=classify model=C:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls_qat-2\weights\best

### Compare FP32, PTQ, and QAT on Test

All three models are scored on the same held-out test split. QAT accuracy gain is reported against the PTQ ONNX model, along with the QAT-vs-FP32 gap and exported sizes.

In [6]:
PTQ_BASELINE_PATH = QAT_SOURCE_MODEL.with_name("best_int8.onnx")
if not PTQ_BASELINE_PATH.is_file():
    raise FileNotFoundError(f"PTQ baseline export not found beside {QAT_SOURCE_MODEL}")
QAT_MODEL_PATH = globals().get("QAT_ONNX_PATH")
if QAT_MODEL_PATH is None:
    raise RuntimeError("Run QAT training and QAT ONNX export before this comparison")
QAT_MODEL_PATH = Path(QAT_MODEL_PATH)


def evaluate_test_accuracy(model_path):
    model = YOLO(str(model_path))
    metrics = model.val(
        data=str(QAT_SPLIT_DATA),
        split="test",
        imgsz=QAT_IMAGE_SIZE,
        batch=QAT_BATCH_SIZE,
        device="cpu",
        workers=0,
    )
    return float(metrics.top1), float(metrics.top5)


FP32_TEST_TOP1, FP32_TEST_TOP5 = evaluate_test_accuracy(QAT_SOURCE_MODEL)
PTQ_TEST_TOP1, PTQ_TEST_TOP5 = evaluate_test_accuracy(PTQ_BASELINE_PATH)
QAT_TEST_TOP1, QAT_TEST_TOP5 = evaluate_test_accuracy(QAT_MODEL_PATH)

FP32_SIZE_MB = QAT_SOURCE_MODEL.stat().st_size / (1024 ** 2)
PTQ_SIZE_MB = PTQ_BASELINE_PATH.stat().st_size / (1024 ** 2)
QAT_SIZE_MB = QAT_MODEL_PATH.stat().st_size / (1024 ** 2)

print(f"FP32 test accuracy: top-1={FP32_TEST_TOP1:.4f}, top-5={FP32_TEST_TOP5:.4f}")
print(f"PTQ test accuracy:  top-1={PTQ_TEST_TOP1:.4f}, top-5={PTQ_TEST_TOP5:.4f}")
print(f"QAT test accuracy:  top-1={QAT_TEST_TOP1:.4f}, top-5={QAT_TEST_TOP5:.4f}")
print(f"QAT gain over PTQ: {QAT_TEST_TOP1 - PTQ_TEST_TOP1:+.4f} top-1")
print(f"QAT gap to FP32:   {QAT_TEST_TOP1 - FP32_TEST_TOP1:+.4f} top-1")
print(f"Model sizes (MiB): FP32={FP32_SIZE_MB:.2f}, PTQ={PTQ_SIZE_MB:.2f}, QAT={QAT_SIZE_MB:.2f}")

Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cu130 CPU (AMD Ryzen 7 7445HS w/ Radeon 740M Graphics)
YOLO26n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 3.2 GFLOPs
train: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\train... found 8581 images in 10 classes  
val: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\val... found 1839 images in 10 classes  
test: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\test... found 1839 images in 10 classes  
WARNING test: Slow image access detected (ping: 0.20.1 ms, read: 17.36.8 MB/s, size: 14.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
test: Scanning C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split\test... 1839 images, 0 corrupt: 100% ━━━

## Fine-Tune with TrashNet++

This adds the 1,344 images from the `Classwise Formet` branch of your local TrashNet++ archive to the existing Garbage Dataset. The YOLO-labeled branch is excluded because this workflow is image classification. `Can` maps to `metal`; `Glass`, `Metal`, and `Plastic` merge with their matching classes; `Cable`, `E-wast`, and `Medical Wast` remain distinct classes. This creates a new 13-class split and a separate fine-tuned model run.

In [14]:
from pathlib import Path

import torch

WORKSPACE = Path.cwd()
GARBAGE_SOURCE = Path.home() / "Downloads" / "Garbage Dataset - Post-Training Quantization (PTQ)" / "standardized_256"
GARBAGE_SPLIT_DATA = Path.home() / "Downloads" / "Garbage Dataset - Post-Training Quantization (PTQ)" / "yolo_cls_256_split"
TRASHNET_CONTAINER = Path.home() / "Downloads" / "TrashNet++ A Real-Time Multi-Class Waste Recogniti"
TRASHNET_ROOT = TRASHNET_CONTAINER / "TrashNet++ A Real-Time Multi-Class Waste Recogniti"
TRASHNET_ARCHIVE = TRASHNET_ROOT / "TrashNet++-jCDBig.zip"
COMBINED_DATA = TRASHNET_ROOT / "garbage_trashnetpp_13class_preserved_holdout"
COMBINED_IMAGE_SIZE = 224
COMBINED_BATCH_SIZE = 32
COMBINED_EPOCHS = 12
COMBINED_SEED = 42
COMBINED_VALIDATION_FRACTION = 0.15
COMBINED_TEST_FRACTION = 0.15
COMBINED_DEVICE = 0 if torch.cuda.is_available() else "cpu"
TRASHNET_CLASS_MAP = {
    "Cable": "cable",
    "Can": "metal",
    "E-wast": "e_waste",
    "Glass": "glass",
    "Medical Wast": "medical_waste",
    "Metal": "metal",
    "Plastic": "plastic",
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_MANIFEST = COMBINED_DATA / "split_manifest.json"

if not GARBAGE_SOURCE.is_dir():
    raise FileNotFoundError(f"Existing garbage dataset not found: {GARBAGE_SOURCE}")
if not (GARBAGE_SPLIT_DATA / "split_manifest.json").is_file():
    raise FileNotFoundError(f"Original held-out split manifest not found: {GARBAGE_SPLIT_DATA}")
if not TRASHNET_ARCHIVE.is_file():
    raise FileNotFoundError(f"TrashNet++ class-folder archive not found: {TRASHNET_ARCHIVE}")

GARBAGE_CLASSES = sorted(path.name for path in GARBAGE_SOURCE.iterdir() if path.is_dir())
COMBINED_CLASS_NAMES = sorted(set(GARBAGE_CLASSES) | set(TRASHNET_CLASS_MAP.values()))
FP32_CHECKPOINTS = [
    path
    for path in (WORKSPACE / "runs" / "classify").glob("garbage_yolo26n_cls*/weights/best.pt")
    if "qat" not in path.parents[1].name.lower()
]
if not FP32_CHECKPOINTS:
    raise FileNotFoundError("No trained FP32 garbage classifier checkpoint was found")
COMBINED_START_MODEL = max(FP32_CHECKPOINTS, key=lambda path: path.stat().st_mtime)

print(f"Base dataset classes: {len(GARBAGE_CLASSES)}")
print(f"Existing holdout split: {GARBAGE_SPLIT_DATA}")
print(f"TrashNet++ classes: {TRASHNET_CLASS_MAP}")
print(f"Combined classes ({len(COMBINED_CLASS_NAMES)}): {COMBINED_CLASS_NAMES}")
print(f"Starting checkpoint: {COMBINED_START_MODEL}")
print(f"Training device: {COMBINED_DEVICE}")
print(f"Combined dataset output: {COMBINED_DATA}")

Base dataset classes: 10
Existing holdout split: C:\Users\nitis\Downloads\Garbage Dataset - Post-Training Quantization (PTQ)\yolo_cls_256_split
TrashNet++ classes: {'Cable': 'cable', 'Can': 'metal', 'E-wast': 'e_waste', 'Glass': 'glass', 'Medical Wast': 'medical_waste', 'Metal': 'metal', 'Plastic': 'plastic'}
Combined classes (13): ['battery', 'biological', 'cable', 'cardboard', 'clothes', 'e_waste', 'glass', 'medical_waste', 'metal', 'paper', 'plastic', 'shoes', 'trash']
Starting checkpoint: c:\Users\nitis\YOLO26\runs\classify\garbage_yolo26n_cls-2\weights\best.pt
Training device: 0
Combined dataset output: C:\Users\nitis\Downloads\TrashNet++ A Real-Time Multi-Class Waste Recogniti\TrashNet++ A Real-Time Multi-Class Waste Recogniti\garbage_trashnetpp_13class_preserved_holdout


In [15]:
import json
import os
import random
import shutil
import tempfile
import zipfile
from collections import Counter

BASE_SPLIT_MANIFEST = GARBAGE_SPLIT_DATA / "split_manifest.json"
base_manifest = json.loads(BASE_SPLIT_MANIFEST.read_text(encoding="utf-8"))
if base_manifest.get("source") != str(GARBAGE_SOURCE) or base_manifest.get("classes") != GARBAGE_CLASSES:
    raise ValueError(f"Original split manifest does not match the source dataset: {BASE_SPLIT_MANIFEST}")

SPLIT_COUNTS = {split: {} for split in ("train", "val", "test")}
SPLIT_COUNTS_BY_CLASS = Counter()
trashnet_counts = Counter()

if COMBINED_DATA.exists():
    if not SPLIT_MANIFEST.is_file():
        raise FileExistsError(f"Refusing to reuse unrecognized folder: {COMBINED_DATA}")
    combined_manifest = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
    if (
        combined_manifest.get("garbage_source") != str(GARBAGE_SOURCE)
        or combined_manifest.get("garbage_split_source") != str(GARBAGE_SPLIT_DATA)
        or combined_manifest.get("trashnet_archive") != str(TRASHNET_ARCHIVE)
        or combined_manifest.get("class_map") != TRASHNET_CLASS_MAP
        or combined_manifest.get("classes") != COMBINED_CLASS_NAMES
        or not combined_manifest.get("preserves_original_test_split")
    ):
        raise ValueError(f"Existing combined split does not match the configured sources: {COMBINED_DATA}")
    SPLIT_COUNTS = combined_manifest["counts"]
    trashnet_counts = Counter(combined_manifest["trashnet_counts"])
    SPLIT_COUNTS_BY_CLASS = Counter({
        class_name: sum(SPLIT_COUNTS[split].get(class_name, 0) for split in ("train", "val", "test"))
        for class_name in COMBINED_CLASS_NAMES
    })
    print(f"Reusing existing combined split: {COMBINED_DATA}")
else:
    image_records = {
        split: {class_name: [] for class_name in COMBINED_CLASS_NAMES}
        for split in ("train", "val", "test")
    }
    for split in ("train", "val", "test"):
        for class_name in GARBAGE_CLASSES:
            source_dir = GARBAGE_SPLIT_DATA / split / class_name
            if not source_dir.is_dir():
                raise FileNotFoundError(f"Expected original {split} folder missing: {source_dir}")
            class_files = [
                path for path in source_dir.iterdir()
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            ]
            expected_count = base_manifest["counts"][split].get(class_name)
            if len(class_files) != expected_count:
                raise ValueError(
                    f"Original split count mismatch for {split}/{class_name}: "
                    f"expected {expected_count}, found {len(class_files)}"
                )
            image_records[split][class_name].extend(
                ("file", path, path.name) for path in class_files
            )

    trashnet_records = {source_class: [] for source_class in TRASHNET_CLASS_MAP}
    COMBINED_DATA.parent.mkdir(parents=True, exist_ok=True)
    build_root = Path(tempfile.mkdtemp(prefix="trashnetpp_holdout_", dir=COMBINED_DATA.parent))
    rng = random.Random(COMBINED_SEED)
    linked_files = 0
    copied_files = 0

    try:
        with zipfile.ZipFile(TRASHNET_ARCHIVE) as archive:
            for entry in archive.infolist():
                prefix = "Classwise Formet/"
                if entry.is_dir() or not entry.filename.startswith(prefix):
                    continue
                relative_path = entry.filename[len(prefix):]
                parts = relative_path.split("/")
                if len(parts) != 2 or Path(parts[1]).suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                source_class, source_name = parts
                if source_class not in TRASHNET_CLASS_MAP:
                    raise ValueError(f"Unexpected TrashNet++ class folder: {source_class}")
                target_name = f"trashnetpp_{source_class.lower().replace(' ', '_')}_{source_name}"
                trashnet_records[source_class].append(("zip", entry.filename, target_name))
                trashnet_counts[source_class] += 1

            if set(trashnet_counts) != set(TRASHNET_CLASS_MAP):
                raise ValueError(f"TrashNet++ classes do not match the expected map: {dict(trashnet_counts)}")

            for source_class, records in trashnet_records.items():
                if len(records) < 3:
                    raise ValueError(f"TrashNet++ class {source_class!r} needs at least 3 images")
                rng.shuffle(records)
                test_count = max(1, round(len(records) * COMBINED_TEST_FRACTION))
                val_count = max(1, round(len(records) * COMBINED_VALIDATION_FRACTION))
                train_end = len(records) - test_count - val_count
                partitions = {
                    "train": records[:train_end],
                    "val": records[train_end:len(records) - test_count],
                    "test": records[len(records) - test_count:],
                }
                target_class = TRASHNET_CLASS_MAP[source_class]
                for split, split_records in partitions.items():
                    image_records[split][target_class].extend(split_records)

            for split in ("train", "val", "test"):
                for class_name, records in image_records[split].items():
                    destination = build_root / split / class_name
                    destination.mkdir(parents=True, exist_ok=True)
                    SPLIT_COUNTS[split][class_name] = len(records)
                    SPLIT_COUNTS_BY_CLASS[class_name] += len(records)
                    for source_kind, source_ref, target_name in records:
                        destination_file = destination / target_name
                        if source_kind == "file":
                            try:
                                os.link(source_ref, destination_file)
                                linked_files += 1
                            except OSError:
                                shutil.copy2(source_ref, destination_file)
                                copied_files += 1
                        else:
                            with archive.open(source_ref) as source_stream, destination_file.open("wb") as target_stream:
                                shutil.copyfileobj(source_stream, target_stream)
                            copied_files += 1

        combined_manifest = {
            "garbage_source": str(GARBAGE_SOURCE),
            "garbage_split_source": str(GARBAGE_SPLIT_DATA),
            "trashnet_archive": str(TRASHNET_ARCHIVE),
            "trashnet_branch": "Classwise Formet",
            "class_map": TRASHNET_CLASS_MAP,
            "classes": COMBINED_CLASS_NAMES,
            "seed": COMBINED_SEED,
            "validation_fraction": COMBINED_VALIDATION_FRACTION,
            "test_fraction": COMBINED_TEST_FRACTION,
            "preserves_original_test_split": True,
            "trashnet_counts": dict(trashnet_counts),
            "counts": SPLIT_COUNTS,
        }
        (build_root / "split_manifest.json").write_text(
            json.dumps(combined_manifest, indent=2), encoding="utf-8"
        )
        build_root.rename(COMBINED_DATA)
    except Exception:
        shutil.rmtree(build_root, ignore_errors=True)
        raise

    print(f"Created holdout-preserving combined split: {COMBINED_DATA}")
    print(f"TrashNet++ images by source class: {dict(trashnet_counts)}")
    print(f"Hard-linked {linked_files:,} original images; copied {copied_files:,} images")

print(f"Split totals: { {split: sum(counts.values()) for split, counts in SPLIT_COUNTS.items()} }")
print(f"Images per combined class: {dict(sorted(SPLIT_COUNTS_BY_CLASS.items()))}")

Created holdout-preserving combined split: C:\Users\nitis\Downloads\TrashNet++ A Real-Time Multi-Class Waste Recogniti\TrashNet++ A Real-Time Multi-Class Waste Recogniti\garbage_trashnetpp_13class_preserved_holdout
TrashNet++ images by source class: {'Cable': 162, 'Can': 100, 'E-wast': 334, 'Glass': 201, 'Medical Wast': 80, 'Metal': 213, 'Plastic': 254}
Hard-linked 12,259 original images; copied 1,344 images
Split totals: {'train': 9523, 'val': 2040, 'test': 2040}
Images per combined class: {'battery': 756, 'biological': 699, 'cable': 162, 'cardboard': 1411, 'clothes': 1892, 'e_waste': 334, 'glass': 1937, 'medical_waste': 80, 'metal': 1243, 'paper': 1336, 'plastic': 1851, 'shoes': 1449, 'trash': 453}


### Fine-Tune the Combined Dataset

This starts from the existing 10-class FP32 checkpoint and remaps matching class-head weights into the combined 13-class model. The original Garbage Dataset test partition stays held out from the checkpoint’s training; only the new TrashNet++ images are split across train/validation/test. The new checkpoint is saved in a separate run folder.

In [16]:
from pathlib import Path

from ultralytics import YOLO

COMBINED_TRAIN_ARGS = {
    "data": str(COMBINED_DATA),
    "epochs": COMBINED_EPOCHS,
    "imgsz": COMBINED_IMAGE_SIZE,
    "batch": COMBINED_BATCH_SIZE,
    "device": COMBINED_DEVICE,
    "workers": 0,
    "patience": 5,
    "optimizer": "AdamW",
    "lr0": 1e-4,
    "lrf": 0.1,
    "cls_pw": 0.25,
    "cls_remap": True,
    "project": str(WORKSPACE / "runs" / "classify"),
    "name": "garbage_trashnetpp_13class_holdout",
}

combined_model = YOLO(str(COMBINED_START_MODEL))
combined_model.train(**COMBINED_TRAIN_ARGS)
combined_trainer = combined_model.trainer
if combined_trainer is None or combined_trainer.save_dir is None:
    raise RuntimeError("Combined-data training finished without an output directory")
COMBINED_BEST_MODEL_PATH = Path(combined_trainer.save_dir) / "weights" / "best.pt"
if not COMBINED_BEST_MODEL_PATH.is_file():
    raise FileNotFoundError(f"Combined-data checkpoint not found: {COMBINED_BEST_MODEL_PATH}")

combined_test_model = YOLO(str(COMBINED_BEST_MODEL_PATH))
combined_test_metrics = combined_test_model.val(
    data=str(COMBINED_DATA),
    split="test",
    imgsz=COMBINED_IMAGE_SIZE,
    batch=COMBINED_BATCH_SIZE,
    device=COMBINED_DEVICE,
    workers=0,
)
print(f"Combined-data best checkpoint: {COMBINED_BEST_MODEL_PATH}")
print(f"Combined test top-1: {float(combined_test_metrics.top1):.4f}")
print(f"Combined test top-5: {float(combined_test_metrics.top5):.4f}")

Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.25, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\nitis\Downloads\TrashNet++ A Real-Time Multi-Class Waste Recogniti\TrashNet++ A Real-Time Multi-Class Waste Recogniti\garbage_trashnetpp_13class_preserved_holdout, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=12, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0

## Structured Channel Pruning and Fine-Tuning

This section starts from the trained 13-class holdout checkpoint and reuses its existing train/validation/test split at 224px. It applies 30% structured pruning to eligible Conv2d channel outputs; the final classifier is excluded as a pruning target, and its class-output dimension is checked before and after pruning.

In [1]:
%pip install torch-pruning

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import torch
import torch.nn as nn
from pathlib import Path

from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv
import torch_pruning as tp

PRUNE_SOURCE_MODEL = Path(r"C:\Users\nitis\YOLO26\runs\classify\garbage_trashnetpp_13class_holdout\weights\best.pt")
PRUNE_DATA = Path.home() / "Downloads" / "Garbage Dataset - Post-Training Quantization (PTQ)" / "yolo_cls_256_split"
PRUNE_IMAGE_SIZE = 224
PRUNE_RATIO = 0.15
PRUNE_BATCH_SIZE = 16
PRUNE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRUNE_OUTPUT_DIR = Path(r"C:\Users\nitis\YOLO26\runs\classify\garbage_trashnetpp_13class_pruned")
PRUNE_FINE_TUNE_EPOCHS = 12
PRUNE_FINE_TUNE_LR = 1e-5

if not PRUNE_SOURCE_MODEL.is_file():
    raise FileNotFoundError(f"Trained checkpoint not found: {PRUNE_SOURCE_MODEL}")
if not PRUNE_DATA.is_dir():
    raise FileNotFoundError(f"Matching dataset split not found: {PRUNE_DATA}")

model = YOLO(str(PRUNE_SOURCE_MODEL))
model.to(PRUNE_DEVICE)

base_model = model.model.model
dummy = torch.randn(1, 3, PRUNE_IMAGE_SIZE, PRUNE_IMAGE_SIZE, device=PRUNE_DEVICE)

print(f"Backbone: {type(base_model).__name__}")
print(f"Classifier head: {type(base_model[-1]).__name__}")
print(f"Stem Conv output channels: {base_model[0].conv.out_channels}")

PRUNE_ORIGINAL_PARAMS = sum(p.numel() for p in model.model.parameters())
PRUNE_ORIGINAL_SIZE_MIB = PRUNE_SOURCE_MODEL.stat().st_size / (1024 ** 2)

# Important: YOLO's classification head is a custom `Classify` wrapper and must stay untouched.
# The default `MagnitudePruner` does not build valid dependency groups for these wrappers, so we
# prune the first valid backbone Conv/BatchNorm pair explicitly with torch-pruning primitives.

def trim_bn_out_channels(bn_layer, prune_idxs):
    if bn_layer is None or not hasattr(bn_layer, "weight"):
        return
    keep_idxs = [i for i in range(bn_layer.num_features) if i not in set(prune_idxs)]
    if len(keep_idxs) == bn_layer.num_features:
        return
    bn_layer.weight.data = bn_layer.weight.data[keep_idxs]
    if bn_layer.bias is not None:
        bn_layer.bias.data = bn_layer.bias.data[keep_idxs]
    bn_layer.running_mean = bn_layer.running_mean[keep_idxs].clone()
    bn_layer.running_var = bn_layer.running_var[keep_idxs].clone()
    bn_layer.num_features = len(keep_idxs)

stem = base_model[0]
next_block = base_model[1]
if not isinstance(stem, Conv) or not isinstance(next_block, Conv):
    raise RuntimeError("Unexpected YOLO stem structure; expected the first two backbone modules to be Conv wrappers.")

# Keep the prune conservative and valid: trim only the stem Conv + matching downstream input channels.
prune_count = max(1, int(stem.conv.out_channels * PRUNE_RATIO))
importance = stem.conv.weight.abs().mean(dim=(1, 2, 3))
prune_idxs = sorted(torch.topk(importance, prune_count, largest=False).indices.cpu().tolist())

print(f"Pruning stem: out_channels={stem.conv.out_channels}, prune_count={len(prune_idxs)}, indices={prune_idxs}")

tp.prune_conv_out_channels(stem.conv, prune_idxs)
trim_bn_out_channels(stem.bn, prune_idxs)
tp.prune_conv_in_channels(next_block.conv, prune_idxs)

# Verify that the pruned network still runs on a dummy forward pass before fine-tuning.
with torch.no_grad():
    results = model(dummy)
    if isinstance(results, (list, tuple)):
        sample = results[0]
    else:
        sample = results
    if hasattr(sample, "probs") and sample.probs is not None:
        logits = sample.probs
    else:
        logits = sample
    if hasattr(logits, "shape"):
        print(f"Dummy forward after pruning: {tuple(logits.shape)}")
    else:
        print("Dummy forward after pruning: successful")

PRUNE_PRUNED_PARAMS = sum(p.numel() for p in base_model.parameters())
if PRUNE_PRUNED_PARAMS >= PRUNE_ORIGINAL_PARAMS:
    raise RuntimeError("Structured pruning did not reduce the tracked backbone parameter count.")

PRUNE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pruned_model_path = PRUNE_OUTPUT_DIR / "weights" / "pruned_initial.pt"
pruned_model_path.parent.mkdir(parents=True, exist_ok=True)
model.model.model = base_model
model.save(str(pruned_model_path))
PRUNE_INITIAL_SIZE_MIB = pruned_model_path.stat().st_size / (1024 ** 2)

print(f"Pruned checkpoint: {pruned_model_path}")
print(f"Parameters before pruning: {PRUNE_ORIGINAL_PARAMS:,}")
print(f"Parameters after pruning:  {PRUNE_PRUNED_PARAMS:,}")
print(f"Model size before pruning: {PRUNE_ORIGINAL_SIZE_MIB:.2f} MiB")
print(f"Model size after pruning:  {PRUNE_INITIAL_SIZE_MIB:.2f} MiB")
print(f"Structured pruning ratio: {PRUNE_RATIO:.0%}")

# Fine-tune the pruned checkpoint for a short low-LR run.
pruned_model = YOLO(str(pruned_model_path))
pruned_model.train(
    data=str(PRUNE_DATA),
    epochs=PRUNE_FINE_TUNE_EPOCHS,
    imgsz=PRUNE_IMAGE_SIZE,
    batch=PRUNE_BATCH_SIZE,
    device=PRUNE_DEVICE,
    workers=0,
    project=str(Path(r"C:\Users\nitis\YOLO26\runs\classify")),
    name="garbage_trashnetpp_13class_pruned",
    lr0=PRUNE_FINE_TUNE_LR,
    lrf=PRUNE_FINE_TUNE_LR,
    patience=6,
    pretrained=False,
    augment=True,
    hsv_h=0.01,
    hsv_s=0.01,
    hsv_v=0.01,
    degrees=5,
    translate=0.05,
    scale=0.05,
    shear=0.0,
    flipud=0.0,
    fliplr=0.5,
)

# Evaluate the fine-tuned pruned model.
fine_tuned_trainer = pruned_model.trainer
if fine_tuned_trainer is None or fine_tuned_trainer.save_dir is None:
    raise RuntimeError("Fine-tuning did not produce a valid trainer object.")

ft_model_path = Path(fine_tuned_trainer.save_dir) / "weights" / "best.pt"
if not ft_model_path.is_file():
    raise FileNotFoundError(f"Fine-tuned model was not produced at {ft_model_path}")

metrics = YOLO(str(ft_model_path)).val(
    data=str(PRUNE_DATA),
    split="test",
    imgsz=PRUNE_IMAGE_SIZE,
    batch=PRUNE_BATCH_SIZE,
    device=PRUNE_DEVICE,
    workers=0,
)

top1 = float(metrics.top1)
top5 = float(metrics.top5)
print(f"Pruned + fine-tuned test top-1 accuracy: {top1:.4f}")
print(f"Pruned + fine-tuned test top-5 accuracy: {top5:.4f}")


Backbone: Sequential
Classifier head: Classify
Stem Conv output channels: 16
Pruning stem: out_channels=16, prune_count=2, indices=[11, 15]

WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.522754669189453. Dividing input by 255.
0: 224x224 e_waste 0.48, metal 0.41, medical_waste 0.04, plastic 0.02, glass 0.02, 8.4ms
Speed: 0.0ms preprocess, 8.4ms inference, 0.7ms postprocess per image at shape (1, 3, 224, 224)
Dummy forward after pruning: (13,)
Pruned checkpoint: C:\Users\nitis\YOLO26\runs\classify\garbage_trashnetpp_13class_pruned\weights\pruned_initial.pt
Parameters before pruning: 1,547,757
Parameters after pruning:  1,547,123
Model size before pruning: 3.07 MiB
Model size after pruning:  3.08 MiB
Structured pruning ratio: 15%
New https://pypi.org/project/ultralytics/8.4.165 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.163  Python-3.12.10 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnosti

AttributeError: 'Sequential' object has no attribute 'yaml'